# Übungen – Block 11: Fehlerbehandlung, Debugging und Tests

Die Übungen arbeiten mit absichtlich eingebauten Fehlern.

## Musterlösung

## Übung 1: Fehler klassifizieren

Ordnen Sie Beispielmeldungen den Kategorien Syntaxfehler, UNREACHABLE, Berechtigung, Variable oder Modulfehler zu.

Beispiele:

- `UNREACHABLE!` → Verbindung
- `Syntax Error while loading YAML` → Syntax
- `VARIABLE IS NOT DEFINED` → Variable
- `Permission denied` bei erreichtem Host → Berechtigung
- Modul meldet ungültigen Parameter → Modulfehler

## Übung 2: Verbosity

Führen Sie einen fehlerhaften Run mit steigender Verbosity aus und notieren Sie, welche Zusatzinformationen hilfreich sind.

```bash
ansible-playbook -i inventory.ini broken.yml -v
ansible-playbook -i inventory.ini broken.yml -vv
ansible-playbook -i inventory.ini broken.yml -vvv
```

Mehr Verbosity sollte gezielt genutzt werden; die höchste Stufe ist nicht automatisch die beste Standardausgabe.

## Übung 3: debug

Geben Sie eine Variable, ein Dictionary und einen Fact mit `debug` aus.

```yaml
- ansible.builtin.debug:
    var: service_port

- ansible.builtin.debug:
    var: server

- ansible.builtin.debug:
    var: ansible_distribution
```

## Übung 4: register

Speichern Sie das Ergebnis eines `command`-Tasks und untersuchen Sie `rc`, `stdout` und `stderr`.

```yaml
- name: Kommando ausführen
  ansible.builtin.command: /bin/echo hello
  register: cmd_result

- ansible.builtin.debug:
    var: cmd_result
```

## Übung 5: failed_when

Definieren Sie eine eigene Fehlerbedingung auf Basis von `rc` oder `stdout`.

```yaml
- name: Prüfen
  ansible.builtin.command: /bin/echo status=bad
  register: result
  failed_when: "'status=bad' in result.stdout"
```

## Übung 6: ignore_errors

Erzeugen Sie einen bewusst tolerierten Fehler und diskutieren Sie, wann `ignore_errors` vertretbar ist.

```yaml
- name: Bewusst tolerierter Test
  ansible.builtin.command: /bin/false
  ignore_errors: true
```

Vertretbar nur, wenn der Fehler fachlich erwartet und der weitere Ablauf trotzdem sinnvoll ist.

## Übung 7: Idempotenz

Führen Sie ein Playbook zweimal aus und untersuchen Sie alle Tasks, die beim zweiten Lauf noch `changed` melden.

```bash
ansible-playbook -i inventory.ini site.yml
ansible-playbook -i inventory.ini site.yml
```

Jeder Task, der im zweiten Lauf noch `changed` meldet, sollte darauf geprüft werden, ob dies fachlich notwendig ist oder aus imperativer Umsetzung resultiert.

## Abschlussübung

Reparieren Sie ein fehlerhaftes Playbook und belegen Sie anschließend mit zwei Läufen, dass es reproduzierbar arbeitet.

Vorgehen:

1. Syntax prüfen.
2. Inventory und Connectivity prüfen.
3. Variablen mit `debug` untersuchen.
4. fehlerhafte Tasks einzeln korrigieren.
5. Playbook vollständig ausführen.
6. denselben Run wiederholen.
7. unerwartete `changed`-Tasks analysieren.

## Ausführliche Fehleranalyse und Bewertung

### Übung 1 – Fehlerkategorien sicher unterscheiden
Ein YAML-Parserfehler entsteht vor der Task-Ausführung; `UNREACHABLE` betrifft den Transport; ein Berechtigungsfehler entsteht häufig erst auf dem Managed Node; undefinierte Variablen werden beim Auswerten der Task sichtbar; ungültige Modulargumente sind Modulfehler. Eine Meldung wie `Permission denied` kann jedoch auch von SSH stammen und dann `UNREACHABLE` bedeuten. Entscheidend sind Task-Phase und vollständiger Fehlerkontext.

### Übung 2 – Verbosity
`-v` zeigt mehr Task-Details, `-vv` zusätzliche Ausführungsinformationen, `-vvv` häufig Verbindungs- und SSH-Details. Höhere Stufen können sensible Informationen offenlegen; Logs entsprechend behandeln.

### Übung 3 – `debug`
`debug: var=server` zeigt die Struktur des Dictionaries, `debug: var=service_port` den konkreten Wert. Facts wie `ansible_facts['distribution']` setzen `gather_facts: true` beziehungsweise einen vorherigen `setup`-Aufruf voraus.

### Übung 4 – `register`
`rc` ist der Rückgabecode (0 ist üblicherweise Erfolg), `stdout` die Standardausgabe und `stderr` die Fehlerausgabe. Bei einem einfachen `command: hostname` sind `stdout` und `rc` erwartbar gesetzt. Shell-Syntax wie `;` oder Umleitungen erfordert `shell` oder einen expliziten Shell-Aufruf; das ist für `hostname` nicht nötig.

### Übung 5 – `failed_when`
Mit `failed_when: "'status=bad' in result.stdout"` wird eine fachliche Fehlerbedingung auch bei `rc=0` ausgelöst. Das eignet sich für Programme, deren Exit-Code fachliche Fehler nicht ausreichend signalisiert. Bedingungen sollten eindeutig und anhand kontrollierter Ausgaben formuliert sein.

### Übung 6 – `ignore_errors`
`ignore_errors: true` erlaubt die Fortsetzung nach einem fehlgeschlagenen Task, beseitigt aber die Ursache nicht und hilft nicht bei allen Fehlerarten (insbesondere nicht allgemein bei nicht erreichbaren Hosts). Sinnvoll ist es nur für erwartete, dokumentierte, nicht kritische Fehlschläge. Sonst besser `block/rescue` oder präzise `failed_when` verwenden.

### Übung 7 und Abschluss – Idempotenz nachweisen
Zuerst Syntax und Verbindung prüfen, dann Fehler korrigieren, Zielzustand kontrollieren und zweimal ausführen. Im zweiten Lauf sind `changed=0`-Werte bei zustandsorientierten Tasks ein gutes Indiz. Ein `command`-Task meldet ohne `changed_when` oft bei jedem Lauf `changed`, obwohl er nichts verändert. Eine sinnvolle Musterlösung unterscheidet daher tatsächliche Zustandsänderung, erwartete Prüfaktion und fehlerhafte Implementierung.